# Transcription pipeline experiments — GridVegetation100mx100m

**PDF**: 3 pages, 6 form-sheets (2 per page).  
**Forms**: M15×2 (p1), M13+L13 (p2), K11+JH-02 (p3).  
**Golden reference**: hand-built for M13 (page 2 top) — see `temp/experiments/multi/gridvegetation100mx100m/golden.xlsx`

Five approaches tested, varying model, thinking, and preprocessing.

**Key question**: Can a direct Gemini API call replace the Textract+codex pipeline — and at what quality/cost tradeoff?

In [ ]:
import json, pathlib, re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BASE = pathlib.Path("../temp/experiments")

## 1. Experiment inventory

In [ ]:
experiments = [
    {
        "id": "E1",
        "label": "3.5-flash\n+thinking",
        "model": "gemini-3.5-flash",
        "thinking": True,
        "textract": False,
        "approach": "Gemini direct-PDF",
        "scope": "3 pages",
        "in_tok": 1881,
        "img_tok": 1873,
        "out_tok": 3901,
        "think_tok": 8795,
        "total_tok": 14577,
        "cost_inr": 13.23,   # empirical billing
        "cost_known": True,
        "elapsed_s": 56.8,
        "forms_found": 6,
        "output": "gemini/gridvegetation100mx100m/output_gemini-3.5-flash.xlsx",
        "raw": "gemini/gridvegetation100mx100m/raw_gemini-3.5-flash.txt",
    },
    {
        "id": "E2",
        "label": "2.5-flash\n+thinking",
        "model": "gemini-2.5-flash",
        "thinking": True,
        "textract": False,
        "approach": "Gemini direct-PDF",
        "scope": "3 pages",
        "in_tok": 1059,
        "img_tok": 0,        # image tokens not separately reported
        "out_tok": 4193,
        "think_tok": 5917,
        "total_tok": 11169,
        "cost_inr": 1.96,
        "cost_known": True,
        "elapsed_s": 49.75,
        "forms_found": 6,
        "output": "gemini/gridvegetation100mx100m/output_gemini-2.5-flash.xlsx",
        "raw": "gemini/gridvegetation100mx100m/raw_gemini-2.5-flash.txt",
    },
    {
        "id": "EA",
        "label": "3.5-flash\nno-think",
        "model": "gemini-3.5-flash",
        "thinking": False,
        "textract": False,
        "approach": "Gemini direct-PDF",
        "scope": "3 pages",
        "in_tok": 1881,
        "img_tok": 1596,
        "out_tok": 4276,
        "think_tok": 0,
        "total_tok": 6157,
        "cost_inr": 0.24,
        "cost_known": True,
        "elapsed_s": 24.69,
        "forms_found": 6,
        "output": "gemini/gridvegetation100mx100m/output_gemini-3.5-flash-nothink.xlsx",
        "raw": "gemini/gridvegetation100mx100m/raw_gemini-3.5-flash-nothink.txt",
    },
    {
        "id": "EB",
        "label": "codex-only\n(gpt-5.5)",
        "model": "gpt-5.5 (codex)",
        "thinking": False,
        "textract": False,
        "approach": "Codex CLI iterative",
        "scope": "3 pages",
        "in_tok": None,      # not broken out by codex
        "img_tok": None,
        "out_tok": None,
        "think_tok": 0,
        "total_tok": 115750,
        "cost_inr": None,    # unknown — billed to OpenAI account
        "cost_known": False,
        "elapsed_s": 360,    # ~6 min
        "forms_found": 6,
        "output": "codex_only/gridvegetation100mx100m/output.xlsx",
        "raw": "codex_only/gridvegetation100mx100m/run.log",
    },
    {
        "id": "EC",
        "label": "3.5-flash\nno-think+v1",
        "model": "gemini-3.5-flash",
        "thinking": False,
        "textract": True,
        "approach": "Gemini + Textract v1.json",
        "scope": "3 pages (v1.json: page 1 only)",
        "in_tok": 29074,
        "img_tok": 1596,
        "out_tok": 4138,
        "think_tok": 0,
        "total_tok": 33212,
        "cost_inr": 0.57,
        "cost_known": True,
        "elapsed_s": 28.7,
        "forms_found": 6,
        "output": "gemini/gridvegetation100mx100m/output_gemini-3.5-flash-nothink-v1.xlsx",
        "raw": "gemini/gridvegetation100mx100m/raw_gemini-3.5-flash-nothink-v1.txt",
    },
    {
        "id": "E0",
        "label": "codex\n+Textract",
        "model": "claude-opus (codex)",
        "thinking": False,
        "textract": True,
        "approach": "Codex CLI + Textract v1.json",
        "scope": "page 1 only (2 forms)",
        "in_tok": None,
        "img_tok": None,
        "out_tok": None,
        "think_tok": 0,
        "total_tok": 89655,
        "cost_inr": None,
        "cost_known": False,
        "elapsed_s": None,   # not recorded
        "forms_found": 2,    # page 1 only
        "output": "multi/gridvegetation100mx100m/output.xlsx",
        "raw": "multi/gridvegetation100mx100m/run.log",
    },
]

# Display summary table
print(f"{'ID':<5} {'Label':<22} {'Scope':<28} {'Tokens':>8} {'Cost (₹)':>10} {'Time':>7} {'Forms':>6}")
print("-" * 95)
for e in experiments:
    cost_str = f"₹{e['cost_inr']:.2f}" if e['cost_known'] else "unknown"
    time_str = f"{e['elapsed_s']:.0f}s" if e['elapsed_s'] else "—"
    print(f"{e['id']:<5} {e['label'].replace(chr(10), ' '):<22} {e['scope']:<28} {e['total_tok']:>8,} {cost_str:>10} {time_str:>7} {e['forms_found']:>6}")

## 2. Cost comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

labels  = [e["label"] for e in experiments]
colors  = ["#e8534b" if e["thinking"] else "#5b9bd5" if not e["textract"] else "#70ad47" for e in experiments]
known   = [e for e in experiments if e["cost_known"]]
unknown = [e for e in experiments if not e["cost_known"]]

# Cost bar chart (known costs)
k_labels = [e["label"] for e in known]
k_costs  = [e["cost_inr"] for e in known]
k_colors = ["#e8534b" if e["thinking"] else "#5b9bd5" if not e["textract"] else "#70ad47" for e in known]

bars = ax1.bar(k_labels, k_costs, color=k_colors, width=0.5, edgecolor="white", linewidth=1.2)
ax1.set_ylabel("Cost (₹ INR)", fontsize=11)
ax1.set_title("Cost per run (INR)", fontsize=12, fontweight="bold")
ax1.set_yscale("log")
ax1.set_ylim(0.1, 30)
for bar, val in zip(bars, k_costs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15,
             f"₹{val:.2f}", ha="center", va="bottom", fontsize=9)

# Unknown cost entries: flat line with annotation
unk_labels = ["codex\n+Textract", "codex-only\n(gpt-5.5)"]
avg_cost   = 85   # rough INR estimate at ~$1 USD avg
ax1.axhline(avg_cost, color="#ff9900", linestyle="--", linewidth=1.5, alpha=0.8)
ax1.text(len(known) - 0.5, avg_cost * 1.15, f"codex runs ~₹{avg_cost} (est.)",
         color="#ff9900", fontsize=8, ha="right")

# Token breakdown (stacked)
all_labels = [e["label"] for e in experiments]
img_toks   = [e["img_tok"] or 0 for e in experiments]
txt_toks   = [(e["in_tok"] or 0) - (e["img_tok"] or 0) for e in experiments]
out_toks   = [e["out_tok"] or 0 for e in experiments]
think_toks = [e["think_tok"] or 0 for e in experiments]
# for codex, total is all we have — show as a single bar
codex_total = [e["total_tok"] if not e["cost_known"] else 0 for e in experiments]

x = np.arange(len(experiments))
ax2.bar(x, img_toks,   color="#5b9bd5", label="Image (input)")
ax2.bar(x, txt_toks,   color="#a9c4e4", bottom=img_toks, label="Text (input)")
btm2 = [i+t for i,t in zip(img_toks, txt_toks)]
ax2.bar(x, out_toks,   color="#70ad47", bottom=btm2, label="Output")
btm3 = [b+o for b,o in zip(btm2, out_toks)]
ax2.bar(x, think_toks, color="#e8534b", bottom=btm3, label="Thinking (billed premium)")
# codex total as hatched
ax2.bar(x, codex_total, color="#ff9900", alpha=0.4, hatch="//", label="Codex total (undifferentiated)")

ax2.set_xticks(x)
ax2.set_xticklabels(all_labels, fontsize=9)
ax2.set_ylabel("Tokens", fontsize=11)
ax2.set_title("Token breakdown", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("cost_tokens.png", dpi=120)
plt.show()

**Key cost observation**: Thinking tokens (red) drove 98% of E1's ₹13.23 bill.
The same model with thinking disabled (EA) costs ₹0.24 — a **55× reduction** with minimal quality loss on clean fields.

EC's extra text tokens (27,478) come entirely from Textract's v1.json payload — 5× more input tokens than EA for a modest ₹0.33 uplift.
Codex runs (E0, EB) cost is unknown here but estimated ~₹85/run at current gpt-5.5 pricing.

## 3. Quality matrix

Scored on M13 (page 2 top) — the form with a hand-built golden.

| Dimension | E1 3.5+think | E2 2.5+think | EA 3.5 no-think | EB codex-only | EC 3.5+v1.json |
|-----------|:---:|:---:|:---:|:---:|:---:|
| **All 6 forms found** | ✅ | ✅ | ✅ | ✅ | ✅ |
| **Metadata (GPS, slope, date)** | ✅ | ⚠️ OCR garble "Mi3" | ✅ | ✅ | ✅ |
| **Absent-species checkboxes** | ✅ | ❌ only Q1 | ✅ | ✅ | ❌ cross-page contamination |
| **Present-species checkboxes** | ⚠️ some gaps | ❌ | ⚠️ some gaps | ⚠️ some gaps | ❌ wrong values |
| **Canopy quadrant grid** | ⚠️ flat list | ❌ | ❌ wrong M comp | ✅ in free-text | ✅ structured sheet |
| **Disturbance field** | ✅ most complete | ⚠️ | ✅ | ✅ | ⚠️ blank |
| **Uncertainty flagging** | ❌ none | ❌ none | ❌ none | ✅ yellow fill | ❌ none |
| **Notation rules (cont. line→blank)** | ❌ not tested | ❌ | ❌ | ✅ JH-02 Q1 correct | ❌ |
| **Crop lineage trail** | ❌ | ❌ | ❌ | ✅ crop_*.png | ❌ |

✅ correct  ⚠️ partial/minor error  ❌ wrong or missing

## 4. Where things broke down — concrete examples

### 4a. Thinking tokens: ₹13 for marginal improvement

In [ ]:
# M13 Alien trees — E1 (thinking) vs EA (no thinking)
# Both read from the same raw JSON outputs

comparison = {
    "field": "M13 Alien trees — Quarter presence/absence",
    "golden": {
        "Silver oak":  ["absent",  "present", "present", "present"],
        "Maesopsis":   ["present", "absent",  "present", "absent"],
        "Spathodea":   ["absent",  "absent",  "absent",  "absent"],
        "Eucalyptus":  ["absent",  "absent",  "absent",  "absent"],
    },
    "E1 (thinking)": {
        "Silver oak":  ["x",  "✓", "✓", "✓"],   # Q1 different notation
        "Maesopsis":   ["✓",  "",  "✓", ""],
        "Spathodea":   ["x",  "x", "x", "x"],
        "Eucalyptus":  ["x",  "x", "x", "x"],
    },
    "EA (no think)": {
        "Silver oak":  ["✓",  "",  "✓", "✓"],   # Q2 blank (E1 has it)
        "Maesopsis":   ["✓",  "",  "✓", "✓"],   # Q4 extra
        "Spathodea":   ["✗",  "✗", "✗", "✗"],
        "Eucalyptus":  ["✗",  "✗", "✗", "✗"],
    },
}

print(f"Field: {comparison['field']}")
print(f"\n{'Species':<14} {'Q1':^10} {'Q2':^10} {'Q3':^10} {'Q4':^10}")
print("-" * 54)
for model in ["golden", "E1 (thinking)", "EA (no think)"]:
    print(f"\n[{model}]")
    for sp, vals in comparison[model].items():
        print(f"  {sp:<14} {vals[0]:^10} {vals[1]:^10} {vals[2]:^10} {vals[3]:^10}")

**Finding**: E1 (thinking, ₹13.23) and EA (no thinking, ₹0.24) differ on only 2 cells out of 16 for this table. The absent species (Spathodea, Eucalyptus) are correct in both. The thinking budget bought Silver oak Q2 and consistent `x`/`✗` notation — not worth ₹13.

### 4b. EC — Textract v1.json cross-page contamination (the worst breakdown)

In [ ]:
# EC fed Gemini a v1.json that only covered PAGE 1 (M15 forms),
# then asked it to read the full 3-page PDF.
# Result: page 2 checkbox reads are badly wrong.

contamination_example = """
M13 Alien trees — Exp EC (Textract v1.json from PAGE 1 fed for all 3 pages)

Species      | Q1   | Q2   | Q3   | Q4   | Expected
-------------|------|------|------|------|----------
Silver oak   | Tick |      | Tick |      | absent / present / present / present
Maesopsis    | Tick |      |      |      | present / absent / present / absent
Spathodea    | Tick | Tick | Tick | Tick | ABSENT in all quarters  ← WRONG
Eucalyptus   | Tick | Tick | Tick | Tick | ABSENT in all quarters  ← WRONG

Root cause: v1.json described page 1's M15 form where Spathodea IS present
in some quarters. Gemini used this as a template when reading page 2,
pattern-matching instead of visually reading the actual checkbox marks.

Also note: EC returned 'Tick' (a word) instead of a mark symbol — the
notation convention broke down entirely on these cells.
"""
print(contamination_example)

**Lesson**: Textract v1.json context must match the page being read. Page-1-only v1.json fed alongside a multi-page PDF causes the model to use the structured page-1 data as a template for all pages. This is worse than no context at all.

### 4c. E2 (2.5-flash) — resolution / OCR quality breakdown

In [ ]:
e2_failures = """
gemini-2.5-flash failures on M13:

1. Grid ID garbled:  read "Mi3" instead of "M13"
   → OCR confusion between '1' and 'i' at lower effective resolution

2. Checkboxes: only Q1 column populated across all species.
   Q2, Q3, Q4 columns blank throughout.
   → Model appears to process PDF at lower visual resolution;
     individual checkbox marks in columns 2-4 not detectable.

3. Canopy grid: completely blank (all four quadrant cells empty)
   → Marginal sketch area not read at all.

Cost: ₹1.96 (7× cheaper than E1, thinking tokens = 5,917)
At this quality level, the cost saving is irrelevant — the output is unusable.
"""
print(e2_failures)

### 4d. EB (codex-only) — notation compliance, but flat structure

In [ ]:
codex_highlights = """
Codex-only strengths (unique to this approach):

1. Continuous-line → blank rule correctly applied for JH-02 (Form 6):
   Q1 column left blank with comment "Q1 has continuous vertical line/no entry"
   → All Gemini variants just left Q1 blank with no explanation.

2. Yellow-flag uncertainty on ambiguous cells:
   - 'Entimena (?)' — uncertain species name in Form 2
   - 'B. oak (?)' — uncertain extra species, Form 2
   - 'Polygonum: L (?)' — uncertain cover value
   - Faint Q4 marks on pages 2-3

3. Cover value for 'Montanoa' Form 6: 'A (?)' — flagged as uncertain
   (all Gemini variants confidently wrote a value without flagging)

Codex-only weaknesses:

1. Flat single-sheet structure (v2) — all 6 forms concatenated vertically.
   Downstream parsing requires knowing row ranges for each form.

2. P/A notation instead of ✓/✗ — inconsistent with Gemini outputs and
   the golden reference. Not wrong, just different.

3. Canopy grid stored as free-text in 'Marginal notes' field:
   "Canopy density sketch: S / S / S / S"
   Correct values, but loses the 2×2 spatial structure.

4. 115,750 tokens, ~6 minutes — 18× more tokens than EA,
   15× more elapsed time.
"""
print(codex_highlights)

### 4e. E0 (codex+Textract) vs EB (codex-only) — scope and per-page efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

runs = [
    ("codex+Textract\n(E0 — page 1 only,\n2 forms)",  89655, 2, "#5b9bd5"),
    ("codex-only\n(EB — all 3 pages,\n6 forms)",      115750, 6, "#70ad47"),
]

for i, (label, total, forms, color) in enumerate(runs):
    ax.bar(i, total, color=color, width=0.4, edgecolor="white")
    ax.text(i, total + 1500, f"{total:,} tokens\n({forms} forms → {total//forms:,}/form)",
            ha="center", va="bottom", fontsize=9)

ax.set_xticks([0, 1])
ax.set_xticklabels([r[0] for r in runs], fontsize=9)
ax.set_ylabel("Total tokens")
ax.set_title("Codex: per-form token efficiency", fontsize=12, fontweight="bold")
ax.set_ylim(0, 145000)

# Per-form efficiency callout
ax.annotate("", xy=(1, 115750/6), xytext=(0, 89655/2),
            arrowprops=dict(arrowstyle="->", color="#e8534b", lw=1.5))
ax.text(0.5, 52000, "tokens/form:\n44,828 → 19,292\n(2.3× more efficient)",
        ha="center", color="#e8534b", fontsize=9)

plt.tight_layout()
plt.savefig("codex_efficiency.png", dpi=120)
plt.show()

**Key insight**: Codex+Textract processed only page 1 (2 forms) at 89,655 tokens — 44,828 tokens/form.
Codex-only processed all 3 pages (6 forms) at 115,750 tokens — 19,292 tokens/form.

The Textract v1.json adds a large structured text payload that the model processes at the start of every run, regardless of how many cells it actually helps. For a 3-page PDF it would cost ~270K tokens in the codex+Textract approach vs 115K for codex-only.

**Textract value assessment**: Near zero for Gemini (actively harmful for multi-page). Marginal for codex — it gave codex a pre-built table structure to cross-reference, which helped checkbox fidelity in the multi-experiment, but codex-only achieved comparable quality with better per-page efficiency.

## 5. Canopy quadrant grid — a recurring challenge across all approaches

In [ ]:
canopy_comparison = """
The form has a 2×2 marginal sketch grid for canopy density and composition:

       NW | NE
       ---|---
       SW | SE

M13 golden: Density = S/S/S/S, Composition = 2*/N/N/N (NW uncertain)

How each approach handled it:

E1 (3.5+think)   : Flat string in Metadata — "S, S, S, S" / "N, N, N, N"
                   Values correct, spatial structure lost.

E2 (2.5+think)   : Completely blank — quadrant area not read.

EA (3.5 no-think): Directional but wrong composition — "N: M, M | S: M, M"
                   Reads N as M for composition. Density correct.

EB (codex-only)  : Free-text marginal note — "S / S / S / S" / "N / N / N / N"
                   Values correct, structure lost, but honest about source.

EC (3.5+v1.json) : BEST STRUCTURE — separate 'M13: Canopy Diagrams' sheet:
                   Diagram Type | NW | NE | SW | SE
                   Density      |  S |  S |  S |  S
                   Composition  |  N |  N |  N |  N
                   Values correct, spatial structure preserved.
                   (Textract's other_text gave it explicit NW/NE/SW/SE labels)

Verdict: EC wins on canopy grid structure, despite being worst on checkboxes.
The per-page v1.json hypothesis: if EC had v1.json for page 2 (not page 1),
it might have both correct checkboxes AND the structured canopy sheet.
"""
print(canopy_comparison)

## 6. Summary scorecard

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

dimensions = [
    "Coverage\n(all forms)",
    "Metadata\naccuracy",
    "Checkbox\naccuracy",
    "Canopy\ngrid",
    "Disturbance\nfield",
    "Uncertainty\nflagging",
    "Notation\ncompliance",
    "Crop\nlineage",
]

# Scores 0-3: 0=fail, 1=partial, 2=good, 3=best
scores = {
    "E1 3.5+think":   [3, 3, 2, 2, 3, 0, 1, 0],
    "E2 2.5+think":   [3, 1, 0, 0, 1, 0, 0, 0],
    "EA 3.5 no-think":[3, 3, 2, 1, 3, 0, 1, 0],
    "EB codex-only":  [3, 3, 2, 2, 3, 3, 3, 3],
    "EC 3.5+v1.json": [3, 3, 0, 3, 1, 0, 0, 0],
}

x = np.arange(len(dimensions))
width = 0.15
exp_colors = ["#e8534b", "#f4b942", "#5b9bd5", "#70ad47", "#9b59b6"]

for i, (label, sc) in enumerate(scores.items()):
    offset = (i - 2) * width
    ax.bar(x + offset, sc, width, label=label, color=exp_colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(dimensions, fontsize=9)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["fail", "partial", "good", "best"])
ax.set_title("Quality by dimension (M13 form)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
ax.set_ylim(0, 3.8)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("quality_scorecard.png", dpi=120)
plt.show()

## 7. Conclusions and next experiment design

### What we learned

**1. Thinking tokens are a cost trap, not a quality investment.**  
EA (₹0.24) vs E1 (₹13.23): 55× cost reduction, 2 cells different out of 16 in the M13 checkbox table. Disable thinking.

**2. Textract v1.json must be page-scoped, or it actively hurts.**  
EC fed page-1 Textract context to a 3-page PDF. Result: page 2-3 checkbox reads are wrong (absent species marked present). The model uses the structured context as a template and pattern-matches instead of reading visually. Rule: either provide per-page v1.json or don't provide it at all.

**3. Codex produces the only reviewable output.**  
Crop artifacts, yellow-flag uncertainty, comment annotations, notation-rule compliance (continuous line → blank). But 115K tokens, ~6 minutes, unknown cost. The quality on hard cells (ambiguous marks, faint Q4 ticks) is genuinely better.

**4. The canopy grid is the one case where Textract helps Gemini.**  
EC's structured canopy sheet (NW/NE/SW/SE) was the best canopy representation. But only because Textract's `other_text` extracted the compass labels explicitly. This is the one retrievable win from the Textract pipeline for Gemini.

**5. E2 (2.5-flash) is not viable.**  
OCR garble on grid IDs, only Q1 column read, blank canopy. Not a cost tradeoff — the output is unusable.

---

### Next experiment

**Base**: EA (3.5-flash, no thinking, pdf-only) — ₹0.24, already done  
**Compare against**: EB (codex-only) — already done

**What to test next (prompt engineering on EA)**:

1. **Checkbox normalization** — add explicit rule: `"represent all checked boxes as 'X', all unchecked boxes as empty string"`. Currently EA uses ✓/✗/α inconsistently.

2. **Uncertainty as structured output** — instead of asking for `(?)` suffix (which Gemini ignores), ask for a separate `"uncertain_cells": [{"sheet": ..., "row": ..., "col": ..., "reason": ...}]` array in the JSON. This is machine-readable and maps directly to yellow-fill.

3. **Bounding boxes per cell** — ask Gemini to emit a `"bbox": [x0,y0,x1,y1]` (fractional) alongside each uncertain value. The review UI can then call `render_page.py` on demand to show the user the exact region the model read — giving the same crop lineage that codex produces natively.

This would let us compare: EA-v2 (prompted Gemini) vs EB (codex) on reviewability, not just accuracy.